## FunctionCall

In [2]:
from langchain_core.messages import HumanMessage, ToolMessage
import dotenv
import os
from langchain_openai import ChatOpenAI
dotenv.load_dotenv()
os.environ['OPENAI_API_KEY'] = os.getenv("DASHSCOPE_API_KEY")
os.environ['OPENAI_BASE_URL'] = os.getenv("BAILIAN_BASE_URL")
# 1.定义工具
tools = [
    {
        "type":"function",
        "function":{
            "name":"get_weather",
            "description":"获取当前城市的天气信息",
            "parameters":{
                "type":"object",
                "properties":{
                    "location":{
                        "type":"string",
                        "description":"城市名字 e.g. 北京"
                    }
                },
                "required":["location"]   
            }
        }
    }
]
# 2.声明模型且绑定工具
model = ChatOpenAI(model = "qwen3.5-flash").bind_tools(tools)
# 3.定义消息体
messages = [HumanMessage(content="深圳的天气怎么样")]
# 4.调用模型
response = model.invoke(messages)
# 5.处理返回的tool_calls
print(response.tool_calls)
calls = response.tool_calls
print(f"function_name:{calls[0]['name']}")
print(f"function_args:{calls[0]['args']['location']}")

def get_weather(location):
    return f"{location} 的天气是下雨天"

#6.拿到方法名以及参数
weather_result = get_weather(calls[0]['args']['location'])
#7.将历史信息追加进去
messages.append(response)
#8将工具返回以及工具ID添加进去
messages.append(ToolMessage(content=weather_result,tool_call_id=calls[0]['id']))
#9.再次调用模型
final_response = model.invoke(messages)
#10.返回结果
print(final_response.content)

[{'name': 'get_weather', 'args': {'location': '深圳'}, 'id': 'call_6e3cbbbb6e4e43c4bb1fd334', 'type': 'tool_call'}]
function_name:get_weather
function_args:深圳
根据最新查询结果，深圳目前正在下雨。建议您出门记得携带雨具，注意防雨保暖。
